In [5]:
import pandas as pd
import numpy as np

# Configuração para visualizar todas as colunas de interesse
pd.set_option('display.max_columns', None)

cols_interesse = [
    'AGEP_A', 'SEX_A', 'EMPWKHRS3_A', 'SLPHOURS_A', 
    'SLPFLL_A', 'SLPSTY_A', 'PAYWORRY_A', 'K6SPD_A',
    'ANXFREQ_A', 'ANXMED_A', 'DEPFREQ_A', 'DEPMED_A'
]

# Leitura direta apenas das variáveis do projeto
df = pd.read_csv("adult24.csv", usecols=cols_interesse)

print(f"Dimensões iniciais: {df.shape[0]:,} linhas e {df.shape[1]} colunas.")
display(df.head(10))

Dimensões iniciais: 32,629 linhas e 12 colunas.


,K6SPD_A,EMPWKHRS3_A,SEX_A,AGEP_A,SLPSTY_A,SLPFLL_A,SLPHOURS_A,DEPMED_A,DEPFREQ_A,ANXMED_A,ANXFREQ_A,PAYWORRY_A
0,2,40.0,1,49,1,1,8,2,5,2,4,2
1,2,40.0,1,53,1,1,8,2,5,2,5,3
2,2,NaN,1,82,1,2,8,2,5,2,5,3
3,2,40.0,1,42,2,1,8,2,4,2,3,1
4,2,45.0,2,38,1,2,8,2,4,1,4,3
5,2,NaN,2,76,3,3,8,2,5,2,4,3
6,2,NaN,1,48,1,1,8,2,5,2,5,2
7,2,50.0,1,51,1,1,6,2,5,1,4,1
8,8,NaN,2,66,8,8,98,2,9,2,9,2
9,2,45.0,1,39,2,2,8,2,3,2,4,3


In [6]:
# Tabela 1: Contagem de nulos e tipos de dados
tab_nulos = pd.DataFrame({
    'Valores_Nulos': df.isnull().sum(),
    'Porcentagem_Nulos (%)': (df.isnull().sum() / len(df) * 100).round(2),
    'Tipo_Dado': df.dtypes
})
print("=== TABELA 1: DIAGNÓSTICO DE VALORES AUSENTES ===")
display(tab_nulos)

# Tabela 2: Resumo estatístico das variáveis contínuas (Idade, Sono e Horas Trabalhadas)
print("\n=== TABELA 2: RESUMO ESTATÍSTICO DAS VARIÁVEIS NUMÉRICAS ===")
display(df[['AGEP_A', 'SLPHOURS_A', 'EMPWKHRS3_A']].describe().T.round(2))

=== TABELA 1: DIAGNÓSTICO DE VALORES AUSENTES ===


,Valores_Nulos,Porcentagem_Nulos (%),Tipo_Dado
K6SPD_A,0,0.0,int64
EMPWKHRS3_A,14618,44.8,float64
SEX_A,0,0.0,int64
AGEP_A,0,0.0,int64
SLPSTY_A,0,0.0,int64
SLPFLL_A,0,0.0,int64
SLPHOURS_A,0,0.0,int64
DEPMED_A,0,0.0,int64
DEPFREQ_A,0,0.0,int64
ANXMED_A,0,0.0,int64



=== TABELA 2: RESUMO ESTATÍSTICO DAS VARIÁVEIS NUMÉRICAS ===


,count,mean,std,min,25%,50%,75%,max
AGEP_A,32629.0,53.45,18.67,18.0,37.0,55.0,69.0,99.0
SLPHOURS_A,32629.0,10.26,16.66,1.0,6.0,7.0,8.0,99.0
EMPWKHRS3_A,18011.0,39.94,13.74,1.0,38.0,40.0,45.0,99.0


In [7]:
# Tabela 3: Distribuição de códigos em variáveis-chave
cols_para_checar = ['SEX_A', 'SLPFLL_A', 'PAYWORRY_A', 'ANXMED_A', 'DEPMED_A', 'ANXFREQ_A']

tab_codigos = pd.DataFrame({col: df[col].value_counts().sort_index() for col in cols_para_checar}).fillna(0).astype(int)
print("=== TABELA 3: CONTAGEM DE CÓDIGOS ORIGINAIS DO QUESTIONÁRIO ===")
display(tab_codigos)

=== TABELA 3: CONTAGEM DE CÓDIGOS ORIGINAIS DO QUESTIONÁRIO ===


,SEX_A,SLPFLL_A,PAYWORRY_A,ANXMED_A,DEPMED_A,ANXFREQ_A
1,14985,14080,4135,4835,4092,4259
2,17639,12577,9576,27260,27978,4698
3,0,2979,18720,0,0,3726
4,0,1926,0,0,0,10156
5,0,0,0,0,0,9178
7,3,22,35,45,48,61
8,0,1023,123,468,483,463
9,2,22,40,21,28,88


In [8]:
# 1. Filtros de sanitização
df = df[df['SLPHOURS_A'] <= 24].copy()
df = df[df['SEX_A'].isin([1, 2])].copy()

for col in ['SLPFLL_A', 'SLPSTY_A', 'PAYWORRY_A', 'ANXMED_A', 'DEPMED_A']:
    df = df[df[col].isin([1, 2, 3, 4])].copy()

for col in ['ANXFREQ_A', 'DEPFREQ_A']:
    df = df[df[col].isin([1, 2, 3, 4, 5])].copy()

# 2. Criação do Target e Rótulos Semânticos
df['target_medicacao'] = ((df['ANXMED_A'] == 1) | (df['DEPMED_A'] == 1)).astype(int)
df['STATUS_MEDICACAO'] = df['target_medicacao'].map({0: 'Não Toma', 1: 'Toma Medicamento'})
df['SEXO_LABEL'] = df['SEX_A'].map({1: 'Masculino', 2: 'Feminino'})
df['INSÔNIA_INICIAL'] = df['SLPFLL_A'].map({1: '1. Nunca', 2: '2. Raramente', 3: '3. Alguns dias', 4: '4. Quase sempre'})
df['ESTRESSE_CONTAS'] = df['PAYWORRY_A'].map({1: '1. Muito Preocupado', 2: '2. Moderado', 3: '3. Pouco', 4: '4. Nada'})
df['FREQ_ANSIEDADE'] = df['ANXFREQ_A'].map({1: '1. Diária', 2: '2. Semanal', 3: '3. Mensal', 4: '4. Rara', 5: '5. Nunca'})

print(f"Dimensões após limpeza: {df.shape[0]:,} linhas válidas.")
display(df[['AGEP_A', 'SEXO_LABEL', 'SLPHOURS_A', 'INSÔNIA_INICIAL', 'ESTRESSE_CONTAS', 'FREQ_ANSIEDADE', 'STATUS_MEDICACAO']].head(10))

Dimensões após limpeza: 31,187 linhas válidas.


,AGEP_A,SEXO_LABEL,SLPHOURS_A,INSÔNIA_INICIAL,ESTRESSE_CONTAS,FREQ_ANSIEDADE,STATUS_MEDICACAO
0,49,Masculino,8,1. Nunca,2. Moderado,4. Rara,Não Toma
1,53,Masculino,8,1. Nunca,3. Pouco,5. Nunca,Não Toma
2,82,Masculino,8,2. Raramente,3. Pouco,5. Nunca,Não Toma
3,42,Masculino,8,1. Nunca,1. Muito Preocupado,3. Mensal,Não Toma
4,38,Feminino,8,2. Raramente,3. Pouco,4. Rara,Toma Medicamento
5,76,Feminino,8,3. Alguns dias,3. Pouco,4. Rara,Não Toma
6,48,Masculino,8,1. Nunca,2. Moderado,5. Nunca,Não Toma
7,51,Masculino,6,1. Nunca,1. Muito Preocupado,4. Rara,Toma Medicamento
9,39,Masculino,8,2. Raramente,3. Pouco,4. Rara,Não Toma
10,63,Masculino,6,1. Nunca,1. Muito Preocupado,3. Mensal,Não Toma


In [9]:
print("=== TABELA 4: PREVALÊNCIA DE MEDICAÇÃO POR GÊNERO (%) ===")
display(pd.crosstab(df['SEXO_LABEL'], df['STATUS_MEDICACAO'], normalize='index').mul(100).round(2))

print("\n=== TABELA 5: PREVALÊNCIA DE MEDICAÇÃO POR NÍVEL DE INSÔNIA (%) ===")
display(pd.crosstab(df['INSÔNIA_INICIAL'], df['STATUS_MEDICACAO'], normalize='index').mul(100).round(2))

print("\n=== TABELA 6: PREVALÊNCIA DE MEDICAÇÃO POR ESTRESSE COM CONTAS (%) ===")
display(pd.crosstab(df['ESTRESSE_CONTAS'], df['STATUS_MEDICACAO'], normalize='index').mul(100).round(2))

print("\n=== TABELA 7: PREVALÊNCIA DE MEDICAÇÃO POR FREQUÊNCIA DE SINTOMAS (%) ===")
display(pd.crosstab(df['FREQ_ANSIEDADE'], df['STATUS_MEDICACAO'], normalize='index').mul(100).round(2))

=== TABELA 4: PREVALÊNCIA DE MEDICAÇÃO POR GÊNERO (%) ===


STATUS_MEDICACAO,Não Toma,Toma Medicamento
SEXO_LABEL,,
Feminino,76.83,23.17
Masculino,88.19,11.81



=== TABELA 5: PREVALÊNCIA DE MEDICAÇÃO POR NÍVEL DE INSÔNIA (%) ===


STATUS_MEDICACAO,Não Toma,Toma Medicamento
INSÔNIA_INICIAL,,
1. Nunca,87.68,12.32
2. Raramente,81.25,18.75
3. Alguns dias,69.96,30.04
4. Quase sempre,64.66,35.34



=== TABELA 6: PREVALÊNCIA DE MEDICAÇÃO POR ESTRESSE COM CONTAS (%) ===


STATUS_MEDICACAO,Não Toma,Toma Medicamento
ESTRESSE_CONTAS,,
1. Muito Preocupado,77.16,22.84
2. Moderado,80.52,19.48
3. Pouco,83.90,16.10



=== TABELA 7: PREVALÊNCIA DE MEDICAÇÃO POR FREQUÊNCIA DE SINTOMAS (%) ===


STATUS_MEDICACAO,Não Toma,Toma Medicamento
FREQ_ANSIEDADE,,
1. Diária,53.35,46.65
2. Semanal,68.18,31.82
3. Mensal,78.31,21.69
4. Rara,88.77,11.23
5. Nunca,96.46,3.54
